# Bonus — Semantic Caching Layer (CSE488 stretch goal, section 4)

Caches full pipeline responses (retrieval + LLM generation) keyed by query *meaning*, not exact text — so a paraphrase of a recent query ("cheap AMD laptop" vs "budget AMD Ryzen laptop") can hit the cache instead of re-running embedding + FAISS + the LLM call.

Benchmarked for latency and hit-rate improvement, as the spec asks.

## 1. Setup — reuses everything from the M4 notebook

In [3]:
# Replace your first setup cell with this:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                 "faiss-cpu", "sentence-transformers", "pandas", "numpy", "pyarrow", 
                 "transformers", "accelerate", "bitsandbytes"])

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-U', 'faiss-cpu', 'sentence-transformers', 'pandas', 'numpy', 'pyarrow', 'transformers', 'accelerate', 'bitsandbytes'], returncode=0)

In [4]:
import time
import numpy as np
import pandas as pd
import faiss
import re
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

df = pd.read_parquet("/kaggle/input/datasets/mrnotalent/laptop-embedding/laptop_chunks_embeddings_with_lineage.parquet")
embeddings = np.stack(df["embedding"].to_numpy()).astype("float32")
device_level = df.drop_duplicates(subset="row_uid").copy()

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

index_flat = faiss.IndexFlatL2(embeddings.shape[1])
index_flat.add(embeddings)

print(f"Dataframe shape: {df.shape}")

# --- Initialize Qwen2.5-7B-Instruct with BitsAndBytesConfig ---
model_name = "Qwen/Qwen2.5-7B-Instruct"

# Set up the 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16 # Optional, but helps with speed/memory on T4s
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto",
    quantization_config=bnb_config
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dataframe shape: (2643, 22)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

## 2. Bring in the M4 pipeline functions

(Same `extract_filters`, `retrieve`, `generate_recommendation` as the M4 notebook — copy them here so this notebook is self-contained.)

In [5]:
def extract_filters(query: str):
    q = query.lower()
    mask = pd.Series(True, index=device_level.index)
    applied = []

    m = re.search(r"under\s*\$?(\d+)", q) or re.search(r"less than\s*\$?(\d+)", q) or re.search(r"budget.*?\$?(\d+)", q)
    if m:
        price = float(m.group(1))
        mask &= device_level["price_usd"] < price
        applied.append(f"price_usd < {price}")

    m = re.search(r"over\s*\$?(\d+)", q) or re.search(r"above\s*\$?(\d+)", q)
    if m:
        price = float(m.group(1))
        mask &= device_level["price_usd"] > price
        applied.append(f"price_usd > {price}")

    m = re.search(r"(\d+)\s*gb\s*ram", q) or re.search(r"at least\s*(\d+)\s*gb", q)
    if m:
        ram = float(m.group(1))
        mask &= device_level["ram_gb"] >= ram
        applied.append(f"ram_gb >= {ram}")

    if any(w in q for w in ["nvidia", "rtx", "geforce", "gaming"]):
        mask &= device_level["gpu"].str.contains("NVIDIA|RTX|GeForce", case=False, na=False)
        applied.append("gpu contains NVIDIA/RTX/GeForce")

    if "amd" in q or "ryzen" in q:
        mask &= device_level["cpu"].str.contains("Ryzen|AMD", case=False, na=False)
        applied.append("cpu contains Ryzen/AMD")

    for cpu_kw in ["i9", "i7", "i5", "i3"]:
        if cpu_kw in q:
            mask &= device_level["cpu"].str.contains(cpu_kw, case=False, na=False)
            applied.append(f"cpu contains {cpu_kw}")

    if "ssd" in q or "nvme" in q:
        mask &= device_level["storage"].str.contains("SSD|PCIe|NVMe", case=False, na=False)
        applied.append("storage is SSD/PCIe/NVMe")

    return mask, applied


def retrieve(query: str, k: int = 5):
    mask, applied_filters = extract_filters(query)
    candidate_uids = set(device_level.loc[mask, "row_uid"])

    if len(candidate_uids) < k:
        candidate_uids = set(device_level["row_uid"])
        applied_filters = applied_filters + ["(filters relaxed \u2014 too few matches)"]

    candidate_chunk_idx = df.index[df["row_uid"].isin(candidate_uids)].to_numpy()
    candidate_embeddings = embeddings[candidate_chunk_idx]

    sub_index = faiss.IndexFlatL2(candidate_embeddings.shape[1])
    sub_index.add(candidate_embeddings)

    query_vec = embed_model.encode([query]).astype("float32")
    _, local_idx = sub_index.search(query_vec, min(k, len(candidate_chunk_idx)))
    global_idx = candidate_chunk_idx[local_idx[0]]

    results = df.iloc[global_idx][
        ["title", "price_usd", "cpu", "ram_gb", "storage", "gpu", "display", "battery", "chunk_text", "lineage"]
    ]
    return results, applied_filters


def generate_recommendation(query: str, retrieved_df: pd.DataFrame):
    context_blocks = []
    for i, row in retrieved_df.iterrows():
        context_blocks.append(
            f"- {row['title']} | ${row['price_usd']:.2f} | {row['cpu']} | {row['ram_gb']}GB RAM | "
            f"{row['storage']} | {row['gpu']} | {row['display']}"
        )
    context = "\n".join(context_blocks)

    prompt = f"""You are a laptop recommendation assistant. Base your answer ONLY on the devices listed below — do not invent specs or devices not present here.

User request: {query}

Retrieved candidate devices:
{context}

Give a short recommendation: pick the best match (or top 2), and justify the choice using only the specs shown above."""

    # Format the prompt using Qwen's required chat template
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Generate the response
    generated_ids = model.generate(
        **model_inputs, 
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True
    )
    
    # Slice the output to exclude the input prompt tokens
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

def recommend_uncached(query: str, k: int = 5):
    """The original, uncached pipeline — kept separate so we can benchmark against it."""
    retrieved, applied_filters = retrieve(query, k)
    answer = generate_recommendation(query, retrieved)
    return {
        "query": query,
        "filters_applied": applied_filters,
        "retrieved_context": retrieved.drop(columns=["lineage"]).to_dict(orient="records"),
        "recommendation": answer,
    }

## 3. The semantic cache

Stores (query embedding, full response) pairs. On a new query, checks cosine similarity against every cached query embedding — above a threshold counts as a "semantic hit" and skips straight to returning the cached response, no retrieval or LLM call needed.

In [6]:
class SemanticCache:
    def __init__(self, similarity_threshold: float = 0.92, max_size: int = 200):
        self.threshold = similarity_threshold
        self.max_size = max_size
        self.embeddings = []   # list of normalized query embeddings
        self.entries = []      # list of (original_query, response_dict)

    def _normalize(self, vec):
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def lookup(self, query_embedding: np.ndarray):
        if not self.embeddings:
            return None, None
        q = self._normalize(query_embedding)
        sims = np.array([np.dot(q, e) for e in self.embeddings])
        best_idx = int(np.argmax(sims))
        best_sim = float(sims[best_idx])
        if best_sim >= self.threshold:
            matched_query, response = self.entries[best_idx]
            return response, {"matched_query": matched_query, "similarity": best_sim}
        return None, None

    def store(self, query: str, query_embedding: np.ndarray, response: dict):
        if len(self.embeddings) >= self.max_size:
            self.embeddings.pop(0)
            self.entries.pop(0)
        self.embeddings.append(self._normalize(query_embedding))
        self.entries.append((query, response))

    def __len__(self):
        return len(self.entries)


cache = SemanticCache(similarity_threshold=0.92)


def recommend_cached(query: str, k: int = 5):
    query_embedding = embed_model.encode([query])[0].astype("float32")

    cached_response, match_info = cache.lookup(query_embedding)
    if cached_response is not None:
        result = dict(cached_response)
        result["cache_hit"] = True
        result["matched_query"] = match_info["matched_query"]
        result["similarity"] = round(match_info["similarity"], 4)
        return result

    result = recommend_uncached(query, k)
    result["cache_hit"] = False
    cache.store(query, query_embedding, result)
    return result

## 4. Benchmark — latency and hit-rate, cached vs uncached

Mix of exact repeats, paraphrases (should hit the cache), and genuinely new queries (should miss).

In [7]:
benchmark_queries = [
    # --- group 1: AMD budget laptop (1 original + 3 paraphrases -> should hit after the first) ---
    "AMD Ryzen laptop under $600",
    "cheap AMD laptop under $600 dollars",
    "budget laptop with AMD Ryzen processor, under 600 bucks",
    "AMD Ryzen laptop under $600",                          # exact repeat -> should hit

    # --- group 2: NVIDIA gaming laptop ---
    "gaming laptop with NVIDIA graphics under $1200",
    "NVIDIA gaming laptop under $1200 budget",
    "cheap gaming laptop with GeForce GPU, under $1200",

    # --- group 3: high RAM / storage workstation ---
    "laptop with 32GB RAM and 1TB SSD",
    "workstation laptop, 32GB memory, 1TB SSD storage",
    "32 gigs of ram and a terabyte of SSD, what laptop fits",

    # --- group 4: lightweight ultrabook ---
    "lightweight ultrabook for travel",
    "thin and light laptop for a college student",
    "portable ultrabook good for carrying around campus",

    # --- group 5: cheapest i9 ---
    "cheapest laptop with an i9 processor",
    "lowest price i9 laptop",
    "budget-friendly Intel i9 laptop option",

    # --- group 6: business laptop ---
    "reliable business laptop under $900",
    "affordable laptop for office work, under $900",
    "budget business laptop, less than 900 dollars",

    # --- genuine one-off misses: no paraphrase pair, should never hit ---
    "2-in-1 convertible touchscreen laptop",
    "laptop with the best battery life for all-day use",
    "laptop good for video editing and Adobe Premiere",
    "MacBook alternative with similar build quality",
    "laptop with a numeric keypad for spreadsheet work",
    "quietest laptop fan noise under load",
]

print(f"benchmark set size: {len(benchmark_queries)} queries "
      f"({len(benchmark_queries) - 6} in paraphrase groups, 6 standalone)")

print("=== Cache-enabled run ===")
cache = SemanticCache(similarity_threshold=0.92)  # fresh cache for a clean benchmark
cached_timings = []
for q in benchmark_queries:
    start = time.perf_counter()
    result = recommend_cached(q)
    elapsed = time.perf_counter() - start
    cached_timings.append({"query": q, "cache_hit": result["cache_hit"], "latency_s": elapsed,
                            "matched_query": result.get("matched_query"), "similarity": result.get("similarity")})
    hit_str = f"HIT (matched: \"{result.get('matched_query')}\", sim={result.get('similarity')})" if result["cache_hit"] else "MISS"
    print(f"[{elapsed:.3f}s] {hit_str:60s} | {q}")

cached_df = pd.DataFrame(cached_timings)
print()
print(f"Cache hit rate: {cached_df['cache_hit'].sum()}/{len(cached_df)} = {cached_df['cache_hit'].mean():.1%}")
print(f"Avg latency on hits:  {cached_df[cached_df['cache_hit']]['latency_s'].mean():.4f}s")
print(f"Avg latency on misses: {cached_df[~cached_df['cache_hit']]['latency_s'].mean():.4f}s")


benchmark set size: 25 queries (19 in paraphrase groups, 6 standalone)
=== Cache-enabled run ===
[43.229s] MISS                                                         | AMD Ryzen laptop under $600
[37.662s] MISS                                                         | cheap AMD laptop under $600 dollars
[39.053s] MISS                                                         | budget laptop with AMD Ryzen processor, under 600 bucks
[0.008s] HIT (matched: "AMD Ryzen laptop under $600", sim=1.0)        | AMD Ryzen laptop under $600
[37.149s] MISS                                                         | gaming laptop with NVIDIA graphics under $1200
[0.008s] HIT (matched: "gaming laptop with NVIDIA graphics under $1200", sim=0.9647) | NVIDIA gaming laptop under $1200 budget
[33.772s] MISS                                                         | cheap gaming laptop with GeForce GPU, under $1200
[19.997s] MISS                                                         | laptop with 32GB RAM 

In [8]:
print("=== Uncached run (same queries, no caching at all) ===")
uncached_timings = []
for q in benchmark_queries:
    start = time.perf_counter()
    result = recommend_uncached(q)
    elapsed = time.perf_counter() - start
    uncached_timings.append({"query": q, "latency_s": elapsed})
    print(f"[{elapsed:.3f}s] {q}")

uncached_df = pd.DataFrame(uncached_timings)
print()
print(f"Avg latency (no cache): {uncached_df['latency_s'].mean():.4f}s")
print(f"Total time (no cache):  {uncached_df['latency_s'].sum():.4f}s")
print(f"Total time (with cache): {cached_df['latency_s'].sum():.4f}s")
speedup = uncached_df["latency_s"].sum() / cached_df["latency_s"].sum()
print(f"Overall speedup from caching: {speedup:.2f}x")

=== Uncached run (same queries, no caching at all) ===
[41.333s] AMD Ryzen laptop under $600
[29.368s] cheap AMD laptop under $600 dollars
[40.929s] budget laptop with AMD Ryzen processor, under 600 bucks
[28.400s] AMD Ryzen laptop under $600
[33.602s] gaming laptop with NVIDIA graphics under $1200
[35.684s] NVIDIA gaming laptop under $1200 budget
[34.728s] cheap gaming laptop with GeForce GPU, under $1200
[27.806s] laptop with 32GB RAM and 1TB SSD
[24.777s] workstation laptop, 32GB memory, 1TB SSD storage
[21.280s] 32 gigs of ram and a terabyte of SSD, what laptop fits
[26.637s] lightweight ultrabook for travel
[31.705s] thin and light laptop for a college student
[22.434s] portable ultrabook good for carrying around campus
[19.020s] cheapest laptop with an i9 processor
[28.161s] lowest price i9 laptop
[32.750s] budget-friendly Intel i9 laptop option
[27.246s] reliable business laptop under $900
[33.476s] affordable laptop for office work, under $900
[31.682s] budget business laptop, 

## 5. Save results for the report

In [9]:
cached_df.to_csv("semantic_cache_benchmark.csv", index=False)

summary = pd.DataFrame([{
    "hit_rate": cached_df["cache_hit"].mean(),
    "avg_latency_hit_s": cached_df[cached_df["cache_hit"]]["latency_s"].mean(),
    "avg_latency_miss_s": cached_df[~cached_df["cache_hit"]]["latency_s"].mean(),
    "total_latency_cached_s": cached_df["latency_s"].sum(),
    "total_latency_uncached_s": uncached_df["latency_s"].sum(),
    "speedup": uncached_df["latency_s"].sum() / cached_df["latency_s"].sum(),
    "similarity_threshold": 0.92,
}])
summary.to_csv("semantic_cache_summary.csv", index=False)
print(summary.to_string(index=False))

from google.colab import files
files.download("semantic_cache_benchmark.csv")
files.download("semantic_cache_summary.csv")

 hit_rate  avg_latency_hit_s  avg_latency_miss_s  total_latency_cached_s  total_latency_uncached_s  speedup  similarity_threshold
     0.08            0.00788           30.550873              702.685846                762.788199 1.085532                  0.92


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Threshold sensitivity (optional, strengthens the report)

`0.92` was a guess. Sweep a few thresholds to show it was chosen deliberately, not arbitrarily — too low risks false-positive hits (returning a wrong cached answer for a meaningfully different query); too high never hits at all.

In [10]:
threshold_results = []
for threshold in [0.80, 0.85, 0.90, 0.92, 0.95, 0.98]:
    test_cache = SemanticCache(similarity_threshold=threshold)
    hits = 0
    for q in benchmark_queries:
        query_embedding = embed_model.encode([q])[0].astype("float32")
        cached_response, match_info = test_cache.lookup(query_embedding)
        if cached_response is not None:
            hits += 1
        else:
            # simulate storing without a real LLM call, to isolate the threshold effect
            test_cache.store(q, query_embedding, {"query": q, "recommendation": "placeholder"})
    threshold_results.append({"threshold": threshold, "hits": hits, "hit_rate": hits / len(benchmark_queries)})

threshold_df = pd.DataFrame(threshold_results)
print(threshold_df.to_string(index=False))
threshold_df.to_csv("semantic_cache_threshold_sweep.csv", index=False)
files.download("semantic_cache_threshold_sweep.csv")

 threshold  hits  hit_rate
      0.80    11      0.44
      0.85     7      0.28
      0.90     4      0.16
      0.92     2      0.08
      0.95     2      0.08
      0.98     1      0.04


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Corrected analysis — real paraphrase similarities + a fair speedup estimate

The naive cross-run comparison in section 4 (cached run vs. a separately-run uncached run) turned out to be confounded: Groq's API latency varied significantly between the two runs for reasons unrelated to caching (rate limiting / server load), inflating the apparent speedup. This section computes the real paraphrase similarities directly, and a fairer speedup estimate using only within-run data.

In [11]:
def cosine_sim(a, b):
    a, b = a / np.linalg.norm(a), b / np.linalg.norm(b)
    return float(np.dot(a, b))

pairs_to_check = [
    ("AMD Ryzen laptop under $600", "cheap AMD laptop under $600 dollars"),
    ("AMD Ryzen laptop under $600", "budget laptop with AMD Ryzen processor, under 600 bucks"),
    ("gaming laptop with NVIDIA graphics under $1200", "NVIDIA gaming laptop under $1200 budget"),
]
similarity_results = []
for q1, q2 in pairs_to_check:
    e1 = embed_model.encode([q1])[0]
    e2 = embed_model.encode([q2])[0]
    sim = cosine_sim(e1, e2)
    similarity_results.append({"query_a": q1, "query_b": q2, "similarity": sim})
    print(f"{sim:.4f}  |  '{q1}'  vs  '{q2}'")

pd.DataFrame(similarity_results).to_csv("semantic_cache_paraphrase_similarities.csv", index=False)

0.7976  |  'AMD Ryzen laptop under $600'  vs  'cheap AMD laptop under $600 dollars'
0.9113  |  'AMD Ryzen laptop under $600'  vs  'budget laptop with AMD Ryzen processor, under 600 bucks'
0.9647  |  'gaming laptop with NVIDIA graphics under $1200'  vs  'NVIDIA gaming laptop under $1200 budget'


In [12]:
# fair speedup: use this run's own miss-latency average as the "uncached" baseline,
# rather than comparing against a separate run (which is confounded by API latency variance)
avg_miss = cached_df[~cached_df["cache_hit"]]["latency_s"].mean()
fair_baseline_total = avg_miss * len(cached_df)
fair_speedup = fair_baseline_total / cached_df["latency_s"].sum()

print(f"Naive cross-run speedup (in summary.csv): {summary['speedup'].iloc[0]:.2f}x  \u2014 CONFOUNDED, do not report as-is")
print(f"Fair within-run speedup estimate: {fair_speedup:.2f}x  \u2014 use this number")

corrected_summary = pd.DataFrame([{
    "hit_rate": cached_df["cache_hit"].mean(),
    "avg_latency_hit_s": cached_df[cached_df["cache_hit"]]["latency_s"].mean(),
    "avg_latency_miss_s": avg_miss,
    "fair_speedup_within_run": fair_speedup,
    "naive_speedup_cross_run_DO_NOT_USE": summary["speedup"].iloc[0],
    "similarity_threshold": 0.92,
}])
corrected_summary.to_csv("semantic_cache_summary_corrected.csv", index=False)
print(corrected_summary.to_string(index=False))

from google.colab import files
files.download("semantic_cache_paraphrase_similarities.csv")
files.download("semantic_cache_summary_corrected.csv")

Naive cross-run speedup (in summary.csv): 1.09x  — CONFOUNDED, do not report as-is
Fair within-run speedup estimate: 1.09x  — use this number
 hit_rate  avg_latency_hit_s  avg_latency_miss_s  fair_speedup_within_run  naive_speedup_cross_run_DO_NOT_USE  similarity_threshold
     0.08            0.00788           30.550873                 1.086932                            1.085532                  0.92


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>